# Exercise 4.4

In [ ]:
from search import *
import random
import numpy as np
import matplotlib.pyplot as plt

class QueensLocalSearch(Problem):
    def __init__(self, n=8):
        self.n = n
        initial = tuple(random.randint(0, n - 1) for _ in range(n))
        super().__init__(initial)

    def actions(self, state):
        return [
            (col, row)
            for col in range(self.n)
            for row in range(self.n)
            if row != state[col]
        ]

    def result(self, state, action):
        col, row = action
        new_state = list(state)
        new_state[col] = row
        return tuple(new_state)

    def conflicts(self, state):
        conflicts = 0
        for i in range(self.n):
            for j in range(i + 1, self.n):
                same_row = state[i] == state[j]
                same_diagonal = abs(state[i] - state[j]) == abs(i - j)
                if same_row or same_diagonal:
                    conflicts += 1
        return conflicts

    def goal_test(self, state):
        return self.conflicts(state) == 0

    def value(self, state):
        max_pairs = self.n * (self.n - 1) // 2
        return max_pairs - self.conflicts(state)


# Steepest-ascent hill climbing with cost tracking
def steepest_hc(problem):
    current = Node(problem.initial)
    steps = 0
    while True:
        neighbors = current.expand(problem)
        steps += len(neighbors)
        if not neighbors:
            return current.state, steps
        best = argmax_random_tie(neighbors, key=lambda node: problem.value(node.state))
        if problem.value(best.state) <= problem.value(current.state):
            return current.state, steps
        current = best


# First-choice hill climbing
def first_choice_hc(problem):
    current = Node(problem.initial)
    steps = 0
    while True:
        actions = list(problem.actions(current.state))
        random.shuffle(actions)
        improved = False
        for action in actions:
            neighbor_state = problem.result(current.state, action)
            steps += 1
            if problem.value(neighbor_state) > problem.value(current.state):
                current = Node(neighbor_state)
                improved = True
                break
        if not improved:
            return current.state, steps


def slow_schedule(t):
    if t > 300:
        return 0
    return 20 * (0.95 ** t)


def run_queens_experiments(num_trials=100):
    results = {
        "steepest":       {"solved": 0, "costs": []},
        "first_choice":   {"solved": 0, "costs": []},
        "random_restart": {"solved": 0, "costs": []},
        "annealing":      {"solved": 0, "costs": []},
    }

    for _ in range(num_trials):
        # Steepest-ascent
        problem = QueensLocalSearch(8)
        result, cost = steepest_hc(problem)
        results["steepest"]["costs"].append(cost)
        if problem.goal_test(result):
            results["steepest"]["solved"] += 1

        # First-choice
        problem = QueensLocalSearch(8)
        result, cost = first_choice_hc(problem)
        results["first_choice"]["costs"].append(cost)
        if problem.goal_test(result):
            results["first_choice"]["solved"] += 1

        # Simulated annealing
        problem = QueensLocalSearch(8)
        result = simulated_annealing(problem, schedule=slow_schedule)
        results["annealing"]["costs"].append(300)
        if problem.goal_test(result):
            results["annealing"]["solved"] += 1

        # Random restart
        solved = False
        total_cost = 0
        for _ in range(10):
            problem = QueensLocalSearch(8)
            result, cost = steepest_hc(problem)
            total_cost += cost
            if problem.goal_test(result):
                solved = True
                break
        results["random_restart"]["costs"].append(total_cost)
        if solved:
            results["random_restart"]["solved"] += 1

    return results

## 8-puzzles problem code

In [ ]:
class PuzzleWithValue(EightPuzzle):
    def value(self, state):
        distance = 0
        goal = (1,2,3,4,5,6,7,8,0)
        for i in range(9):
            if state[i] == 0:
                continue
            goal_index = goal.index(state[i])
            x1, y1 = divmod(i, 3)
            x2, y2 = divmod(goal_index, 3)
            distance += abs(x1 - x2) + abs(y1 - y2)
        return -distance


def random_puzzle(moves=50):
    problem = EightPuzzle((1,2,3,4,5,6,7,8,0))
    state = problem.initial
    for _ in range(moves):
        actions = problem.actions(state)
        state = problem.result(state, random.choice(actions))
    return state


def get_optimal_cost(state):
    problem = EightPuzzle(state)
    result = astar_search(problem)
    if result is None:
        return None
    return len(result.solution())


def puzzle_schedule(t):
    if t > 300:
        return 0
    return 10 * (0.95 ** t)


def run_puzzle_experiments(num_trials=50):
    results = []
    for _ in range(num_trials):
        state = random_puzzle()
        optimal_cost = get_optimal_cost(state)
        if optimal_cost is None:
            continue

        # Steepest hill climbing
        problem = PuzzleWithValue(state)
        hc_result, hc_cost = steepest_hc(problem)
        hc_solved = problem.goal_test(hc_result)

        # First-choice hill climbing
        problem = PuzzleWithValue(state)
        fc_result, fc_cost = first_choice_hc(problem)
        fc_solved = problem.goal_test(fc_result)

        # Simulated annealing
        problem = PuzzleWithValue(state)
        sa_result = simulated_annealing(problem, schedule=puzzle_schedule)
        sa_solved = problem.goal_test(sa_result)

        # Random restart
        rr_solved = False
        rr_cost = 0
        for _ in range(10):
            problem = PuzzleWithValue(state)
            rr_result, cost = steepest_hc(problem)
            rr_cost += cost
            if problem.goal_test(rr_result):
                rr_solved = True
                break

        results.append({
            "optimal_cost": optimal_cost,
            "hc_solved": hc_solved, "hc_cost": hc_cost,
            "fc_solved": fc_solved, "fc_cost": fc_cost,
            "sa_solved": sa_solved,
            "rr_solved": rr_solved, "rr_cost": rr_cost,
        })
    return results

In [ ]:
queens_results = run_queens_experiments(100)
puzzle_results = run_puzzle_experiments(50)

In [ ]:
print("=== 8-Queens Results ===")
labels = {"steepest": "Steepest HC", "first_choice": "First-Choice HC", "random_restart": "Random Restart", "annealing": "Simulated Annealing"}
for k, v in queens_results.items():
    print(f"{labels[k]}: {v['solved']/100:.2f} success rate, avg cost: {np.mean(v['costs']):.1f}")

print("\n=== 8-Puzzle Results ===")
n = len(puzzle_results)
hc_success = sum(1 for r in puzzle_results if r["hc_solved"]) / n
fc_success = sum(1 for r in puzzle_results if r["fc_solved"]) / n
sa_success = sum(1 for r in puzzle_results if r["sa_solved"]) / n
rr_success = sum(1 for r in puzzle_results if r["rr_solved"]) / n
opt_avg    = np.mean([r["optimal_cost"] for r in puzzle_results])

print(f"Steepest HC success: {hc_success:.2f}, avg cost: {np.mean([r['hc_cost'] for r in puzzle_results]):.1f}")
print(f"First-Choice HC success: {fc_success:.2f}, avg cost: {np.mean([r['fc_cost'] for r in puzzle_results]):.1f}")
print(f"Simulated Annealing success: {sa_success:.2f}")
print(f"Random Restart success: {rr_success:.2f}, avg cost: {np.mean([r['rr_cost'] for r in puzzle_results]):.1f}")
print(f"A* optimal avg cost: {opt_avg:.1f}")

In [ ]:
alg_labels = ["Steepest HC", "First-Choice HC", "Random Restart", "Sim. Annealing"]
keys = ["steepest", "first_choice", "random_restart", "annealing"]

q_rates = [queens_results[k]['solved'] / 100 for k in keys]
q_costs = [np.mean(queens_results[k]['costs']) for k in keys]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(alg_labels, q_rates)
axes[0].set_title("Figure 1. 8-Queens Success Rate")
axes[0].set_ylabel("Success Rate")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(alg_labels, q_costs)
axes[1].set_title("Figure 2. 8-Queens Avg Search Cost")
axes[1].set_ylabel("Avg Steps")
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

puz_rates = [hc_success, fc_success, rr_success, sa_success]
cost_vals = [
    np.mean([r['hc_cost'] for r in puzzle_results]),
    np.mean([r['fc_cost'] for r in puzzle_results]),
    np.mean([r['rr_cost'] for r in puzzle_results]),
    opt_avg
]
cost_labels = ["Steepest HC", "First-Choice HC", "Random Restart", "A* Optimal"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(alg_labels, puz_rates)
axes[0].set_title("Figure 3. 8-Puzzle Success Rate")
axes[0].set_ylabel("Success Rate")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis='x', rotation=15)
axes[1].bar(cost_labels, cost_vals)
axes[1].set_title("Figure 4. 8-Puzzle Avg Cost vs A* Optimal")
axes[1].set_ylabel("Avg Steps")
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.show()

## Discussion & Conclusion

The experiments reveal clear differences between local search algorithms across both problems.

**8-Queens:** Steepest-ascent hill climbing solves only around 14% of instances — it gets stuck at local maxima where no single queen move improves the board. First-choice hill climbing achieves a similar success rate but with lower search cost, since it stops evaluating neighbors as soon as it finds an improvement instead of checking all 56 possible moves. Random restart dramatically improves success rate (typically above 90%) by re-running hill climbing from new random starting positions, effectively sampling many regions of the state space. Simulated annealing achieves moderate success — it can escape local maxima by occasionally accepting worse moves, but performance depends heavily on the temperature schedule.

**8-Puzzle:** All local search algorithms perform poorly compared to A* search. The puzzle landscape is full of local maxima where no single tile slide reduces the Manhattan distance. Once stuck, hill climbing variants cannot recover. Random restart helps marginally but since the initial state is fixed, restarts do not help as much as in the 8-Queens case. Simulated annealing also struggles — the 8-Puzzle requires a precise sequence of moves that random perturbations rarely produce. A* search, by contrast, always finds the optimal solution and with a much lower average move count, demonstrating that path-based search is fundamentally better suited for problems where the solution is a sequence rather than a configuration.